# Project Overview
This retrospective study examines cardiovascular (CV) risk assessment and toxicity monitoring in 239 advanced prostate cancer patients treated with novel hormonal therapies (abiraterone, enzalutamide, apalutamide, and darolutamide). Using data extracted from UCI’s Epic electronic medical record system (64) variables per patient), the project aims to evaluate how consistently providers assess baseline CV risk factors, identify which factors predict post-treatment CV toxicities, and determine patterns in antihypertensive use before and after therapy initiation. The ultimate goal is to leverage data analytics and AI to identify high-risk patients, develop clear infographics to visualize findings, and potentially implement an automated referral pathway to cardio-oncology within Epic.

## Import Required Libraries and Load Raw Data

In [1]:
import pandas as pd
import os

excel_file = os.path.join('..', 'data', 'raw', 'Cardio_onc_prostate_students.xlsx')
df = pd.read_excel(excel_file)

## Remove Metadata Row

The second row of the dataset contained encoding information describing categorical variable mappings (1-Yes, 2-No, 3-Not Recorded). As this row doesn't represent patient-level observations, it is removed.

### Before Removal

In [2]:
df.head(3)
df.columns

Index(['Unique Patient ID', 'Ethnicity',
       'NHT Prior authorization date (Abiraterone, Enzalutamide, Apalutamide, Darolutamide)',
       'Start date of NHT (Abiraterone, Enzalutamide, Apalutamide, Darolutamide)',
       'BMI (when they started NHT) ', 'Specific NHT used', 'Age at NHT start',
       'Start date of ADT (Lupron, Firmagon, Orgovyx, or Bicalutamide)',
       'Specific ADT agent used', 'Hx of smoking?',
       'Family Hx of heart disease? ', 'Hx of HTN prior to NHT start?', 'SBP',
       'DBP', 'BP medication within 3 months prior to NHT start',
       'On Beta-blocker prior to NHT? Acebutolol, Atenolol, Bisoprolol, Carvedilol, Metoprolol, Nadolol, Nebivolol, Propranolol, Betaxolol, Esmolol, Nebivolol, Carteolol, Penbutolol, Pindolol, Labetalol, Sotalol ',
       'On ACE-I or ARB Prior to NHT? (1- yes, 2- no)Benazepril, Enalapril, Fosinopril, Lisinopril, Ramipril, Candesartan, Irbesartan, Losartan, Olmesartan, Telmisartan, Valsartan',
       'Starting new BP medications

### After Removal

In [3]:
df = df.drop(index=0).reset_index(drop=True)
print(df.head(3))

   Unique Patient ID Ethnicity  \
0                1.0       NaN   
1                2.0       NaN   
2                3.0         1   

  NHT Prior authorization date (Abiraterone, Enzalutamide, Apalutamide, Darolutamide)  \
0                                2022-01-09 00:00:00                                    
1                                2022-01-11 00:00:00                                    
2                                2022-01-12 00:00:00                                    

  Start date of NHT (Abiraterone, Enzalutamide, Apalutamide, Darolutamide)  \
0                                                NaT                         
1                                                NaT                         
2                                         2022-02-01                         

   BMI (when they started NHT)  Specific NHT used  Age at NHT start  \
0                           NaN               NaN               NaN   
1                           NaN               NaN  

## Inspect Data Types and Structure

In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 64 columns):
 #   Column                                                                                                                                                                                                                          Non-Null Count  Dtype         
---  ------                                                                                                                                                                                                                          --------------  -----         
 0   Unique Patient ID                                                                                                                                                                                                               239 non-null    float64       
 1   Ethnicity                                                                                                                

240 Total Rows and 64 Total Columns of many different types. Potentialy column type conversion could be good for future modeling. Further inspection is required.

In [23]:
# Conversion

## Missing Data

In [24]:
missing_summary = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'null_percentage': df.isnull().mean() * 100
}).sort_values(by='null_count', ascending=False)
print(missing_summary)

                                                    null_count  \
date of carotid art disease                                236   
date of PVD/PAD                                            236   
EF                                                         234   
date of HF                                                 234   
date of CVA                                                234   
...                                                        ...   
Specific NHT used                                            7   
NHT Prior authorization date (Abiraterone, Enza...           3   
10-Year ASCVD Risk                                           3   
Prescribing Provider                                         3   
Unique Patient ID                                            1   

                                                    null_percentage  
date of carotid art disease                               98.333333  
date of PVD/PAD                                           98.333333

In [25]:
high_null_cols = missing_summary[missing_summary['null_percentage'] > 50]
print(high_null_cols)

                                          null_count  null_percentage
date of carotid art disease                      236        98.333333
date of PVD/PAD                                  236        98.333333
EF                                               234        97.500000
date of HF                                       234        97.500000
date of CVA                                      234        97.500000
date of MI orstent                               233        97.083333
date of arrhythmia                               229        95.416667
date of CAD                                      222        92.500000
A1c                                              219        91.250000
QTc                                              207        86.250000
10-Yr Risk Score %                               199        82.916667
LDL                                              190        79.166667
HDL                                              190        79.166667
TC                  

Variables with more than 60% missing data will be evaluated for exclusion.

## Clean and Standardize Column Names

In [26]:
# Strip them, lower case, and replace spaces with underscores
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df.head(3)

,unique_patient_id,ethnicity,"nht_prior_authorization_date_(abiraterone,_enzalutamide,_apalutamide,_darolutamide)","start_date_of_nht_(abiraterone,_enzalutamide,_apalutamide,_darolutamide)",bmi_(when_they_started_nht),specific_nht_used,age_at_nht_start,"start_date_of_adt_(lupron,_firmagon,_orgovyx,_or_bicalutamide)",specific_adt_agent_used,hx_of_smoking?,...,saw_cards_after_nht_start?,referred_to_cards_post_nht_start_by_oncology?,documented_counseling_about_diet_by_oncology_w/i_3_mos_of_nht_start?,documented_counseling_about_exercise_by_oncology_w/i_3_mos_of_nht_start?,order_for_echo_w/i_3_mos_of_nht_start?,ecg_w/i_3_month_of_nht_start?,qtc,actively_seen_by_non-oncology_providers?,"has_pcp_(1:no,_2:_at_uci_3:_outside).1",prescribing_provider
0,1.0,NaN,2022-01-09 00:00:00,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MEI, ERIKA A"
1,2.0,NaN,2022-01-11 00:00:00,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MEI, ERIKA A"
2,3.0,1,2022-01-12 00:00:00,2022-02-01,25.99,Darolutamide,73.0,2018-01-26 00:00:00,Lupron,1,...,2,2,2,2,2,NaN,NaN,NaN,2.0,"MEI, ERIKA A"


As we can see, there are still parantheses that contain more metadata like: "has PCP (1:No, 2: at UCI, 3: Outside)". Other column names also have metadata, but is not enclosed within parantheses such as "On Beta-blocker prior to NHT? Acebutolol, Atenolol, Bisoprolol, Carvedilol, Metoprolol, Nadolol, Nebivolol, Propranolol, Betaxolol, Esmolol, Nebivolol, Carteolol, Penbutolol, Pindolol, Labetalol, Sotalol"

## Feature Viability

Specified feature categories for questions of interest! -> done later

## Summary and Next Steps

In [ ]:
# Save preprocessed dataframe to csv
#df.to_csv('preprocessed_cardio_onc.csv', index=False)

After initial preprocessing, 